In [1]:
import pandas as pd
import re
from konlpy.tag import Okt
from collections import defaultdict
# 데이터 로드
df = pd.read_csv('daum_movie_review.csv')
okt = Okt()

In [13]:
# 명사 기반 검색엔진 구현(형태소 분석기 응용) - 형태소 분석기를 통해서 조사가 무엇이든 간에 핵심의미 (명사)

# 역색인 inverted index
inverted_index = defaultdict(list)
for idx, row in df.head(1000).iterrows():
    review = row['review']
    nouns = okt.nouns(review)
    for noun in set(nouns):
        if len(nouns) > 1:
            inverted_index[noun].append(idx)

def search_movie_review(query):
    """질의어에서 명사 추출"""
    query_nouns = okt.nouns(query)
    # 첫 번째 명사 기준으로 검색(단순화)
    target_noun = query_nouns[0]
    if target_noun in inverted_index:
        match_index = inverted_index[target_noun]
        return df.loc[match_index][['review', 'rating']].head()
    else:
        return "검색 결과 없습니다."
    
search_movie_review('영화가')

,review,rating
21,롱턱 타노스의 장갑이 참 맘에 듬. 아이언 맨과 토르 닥터만 생고생하고.. 가...,6
26,이 영화를 보고나서 예전 영화 왓치맨이 생각나더군요 평화를 위해선 불특정 다수의 희...,9
34,어벤져스 답게 쏟아붓긴 했는데 한방이 없었다마블 영화들은 보고나면 늘 긴 예고편 본...,5
41,이건 뭐~ 이 정도 돈과 출연진 가지고 이렇게 망칠 수도 있구나를 보여주는 대표작...,2
44,마블 팬이라면 반드시 봐야하는 영화,10


In [16]:
df['review'][1]

'몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.'

In [17]:
# TTR 응용 고유 토큰 수 / 전체 토큰 수
def calc_ttr(text):
    tokens = okt.morphs(text)
    if not tokens: return 0
    return len(set(tokens)) / len(tokens)

spam_review = '추천합니다 추천합니다 추천합니다 추천합니다 추천합니다'
normal_review = '몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.'
calc_ttr(spam_review), calc_ttr(normal_review)

(0.2, 0.8571428571428571)

In [18]:
# review 데이터 중에서 TTR이 0.4 이하인 리뷰들만 추출
import pandas as pd
import re
from konlpy.tag import Okt

# 데이터 로드
df = pd.read_csv('daum_movie_review.csv')

reviews = df['review']

# 한글만 남기기
cleaned_reviews = [
    re.sub(r'[^가-힣\s]', '', review)
    for review in reviews
]

okt = Okt()

low_ttr_reviews = []

for review in cleaned_reviews:

    # 형태소 분석
    tokens = [
        token
        for token, pos in okt.pos(review)
        if pos in ['Noun', 'Verb', 'Adjective']
        and len(token) >= 2
    ]

    # 토큰이 없는 경우 방지
    if len(tokens) == 0:
        continue

    # TTR 계산
    ttr = len(set(tokens)) / len(tokens)

    # 조건
    if ttr <= 0.4:
        low_ttr_reviews.append(review)

print(low_ttr_reviews[:5])

['어벤져스어벤져스어벤져스', '마블마블마블마블마블', '마블마블마블', '지지고볶고 지지고볶고 지지고볶고 지지고볶고 지지고볶고 지지고볶고 지지고볶고 지지고볶고 지지고볶고 지지고볶고 지지고볶고 지지고볶고 지지고볶고 지지고볶고 지지고볶고 지지고볶고', '쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡쵹챡']


In [20]:
# review데이터 중에서 TTR이 0.4이하인 리뷰들만 추출
import re
def extract_hangule_blank(text):
    return re.sub(r'[^가-힣\s]','',text)

# df['clean_review'] = df['review'].apply(lambda x : extract_hangule_blank(x))
df['clean_review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣\s]','',x))

In [21]:
df['ttr'] = df['clean_review'].apply(lambda x : calc_ttr(x) )

In [25]:
df[df['ttr'] <= 0.4]

,review,rating,date,title,clean_review,ttr
89,good,9,2018.08.04,인피니티 워,,0.0
103,ㅍㅎㅎㅎ,1,2018.08.02,인피니티 워,,0.0
147,................................!!!!!!!!!!!!!!...,0,2018.06.17,인피니티 워,,0.0
162,dksqhkwjdy,1,2018.06.06,인피니티 워,,0.0
167,ㅋㅋㅋ,9,2018.06.05,인피니티 워,,0.0
...,...,...,...,...,...,...
14051,remember me,9,2018.03.04,코코,,0.0
14248,...,9,2018.02.04,코코,,0.0
14317,ㅜ ㅜ,10,2018.01.29,코코,,0.0
14499,Good,10,2018.01.19,코코,,0.0
